In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer

c:\Users\User\Desktop\code\atlas-ai\ml-service\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [35]:
df = pd.read_csv('data/cases.csv')
y = df["label"]

In [36]:
emb_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
X = emb_model.encode(df["text"].to_list())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6865.85it/s]


In [37]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold

grid = GridSearchCV(
    LogisticRegression(max_iter=5000, class_weight="balanced"),
    {"C": [1, 10, 100, 1000]},
    cv=StratifiedKFold(5, shuffle=True, random_state=0),
    scoring="f1_macro",
)
grid.fit(X_train, y_train)
print(grid.best_params_, grid.best_score_)
lr_model = grid.best_estimator_
pred = lr_model.predict(X_test)

{'C': 1000} 0.9404790966916006


In [39]:
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

print(classification_report(y_test, pred))
print(accuracy_score(y_test, pred))

                     precision    recall  f1-score   support

appliance_condition       1.00      0.94      0.97        17
       cancellation       1.00      1.00      1.00        13
   general_question       0.98      0.98      0.98        41
          off_topic       1.00      0.94      0.97        16
    return_exchange       0.93      1.00      0.96        27
    service_request       1.00      1.00      1.00        13

           accuracy                           0.98       127
          macro avg       0.98      0.98      0.98       127
       weighted avg       0.98      0.98      0.98       127

0.9763779527559056


In [40]:
max_len = len(pred) 
y_test_arr = y_test.values

for i in range(max_len): 
    if y_test_arr[i] != pred[i]: 
        er = i 
        print(f"Index: {i}") 
        print(f"Predicted: {pred[i]}") 
        print(f"Actual: {y_test_arr[i]}")
        print("--------------------------")

print("")
probs = lr_model.predict_proba(X_test)
for i in range(len(pred)):
    if y_test.values[i] != pred[i]:
        text = df.loc[y_test.index[i], "text"]
        print(f"{y_test.values[i]} -> {pred[i]} ({probs[i].max():.2f}): {text}")
        print(f"appliance_condition: {probs[i][0]:.2f}")
        print(f"cancellation: {probs[i][1]:.2f}")
        print(f"general_question: {probs[i][2]:.2f}")
        print(f"off_topic: {probs[i][3]}")
        print(f"return_exchange: {probs[i][4]:.2f}")
        print(f"service_request: {probs[i][5]:.2f}")
        print("")

Index: 22
Predicted: return_exchange
Actual: appliance_condition
--------------------------
Index: 32
Predicted: return_exchange
Actual: general_question
--------------------------
Index: 94
Predicted: general_question
Actual: off_topic
--------------------------

appliance_condition -> return_exchange (0.68): the washer is leaking and I want to return it
appliance_condition: 0.32
cancellation: 0.00
general_question: 0.00
off_topic: 7.074992254274548e-07
return_exchange: 0.68
service_request: 0.00

general_question -> return_exchange (0.53): can I return an appliance if I already opened the box
appliance_condition: 0.00
cancellation: 0.00
general_question: 0.47
off_topic: 9.415025203907135e-08
return_exchange: 0.53
service_request: 0.00

off_topic -> general_question (0.81): how many ounces are in a pound
appliance_condition: 0.00
cancellation: 0.00
general_question: 0.81
off_topic: 0.18885394930839539
return_exchange: 0.00
service_request: 0.00



In [34]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sklearn.model_selection import train_test_split

# split on row indices so texts, vectors and labels stay aligned
idx = np.arange(len(df))
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=2)  # same args as your original split
texts_train = df["text"].iloc[idx_train].to_list()

# sanity check: if this fails, the split differs from your original one
assert list(df["label"].iloc[idx_train]) == list(y_train)

# nearest training rows
q = emb_model.encode(["can you debug my javascript function"])
sims = cosine_similarity(q, X_train)[0]
for i in np.argsort(sims)[::-1][:5]:
    print(round(sims[i], 3), y_train[i], texts_train[i])

# other code questions
tests = ["debug my python script", "why does my react component not render",
         "can you fix this sql query", "write a regex for emails"]
print(list(zip(tests, lr_model.predict(emb_model.encode(tests)))))

0.248 return_exchange the dryer is dead, it won't turn on at all
0.245 service_request oven shows an error code and won't turn on, installed yesterday
0.222 general_question I'm trying to cancel my order on the website but it won't let me, can you do it
0.213 return_exchange what are the steps if my dryer is defective on arrival
0.208 general_question freezer stopped working after 2 days, everything thawed
[('debug my python script', 'off_topic'), ('why does my react component not render', 'off_topic'), ('can you fix this sql query', 'off_topic'), ('write a regex for emails', 'off_topic')]


In [32]:
map = {"cancellation": 0, "appliance_condition": 0, "service_request": 0, "return_exchange": 0, "no_topic": 0, "general_question": 0}
for i in range(len(df)):
    chars = set(df.loc[i, "text"].split())
    if "range" in chars:
        map[df.loc[i, "label"]] += 1

print(map)

{'cancellation': 3, 'appliance_condition': 5, 'service_request': 3, 'return_exchange': 5, 'no_topic': 0, 'general_question': 3}
